# GLCMaeV9 Inference Pipeline

Load a trained `GLCMaeV9` checkpoint, run inference on welding robot data, and produce anomaly verdicts with p99 aggregation.

**V9 changes vs V8:**
- **p99 aggregation** (was p95) — captures short-duration anomalies better
- **No tail window** — only stride-based windowing (T=512, seq_len=256, stride=64 → 5 windows)
- **k_skip=1** — minimal boundary trimming
- **GlobalSeqNorm** running stats from checkpoint, used in eval mode

**Threshold calibration flow:**
- Use `threshold_calibration_welding_robot.csv` (normal + 5 anomaly types, labelled)
- Plot score distribution per sensor × per anomaly type
- Select threshold → save to `THRESHOLD_CONFIG` → `threshold_config.json`
- Final evaluation on `test_welding_robot.csv`

**Pipeline:**
1. Load trained model checkpoint
2. Load threshold calibration CSV → preprocess into file-windows
3. Load test CSV + test_labels → preprocess into file-windows
4. Score threshold_calibration + test sets with `Reconf9LocalPreResult`
5. Plot score distributions → select threshold
6. Apply threshold to test set → evaluate per-type F1 / AUROC
7. Per-sensor heatmap analysis
8. Summary report

In [ ]:
# %% Imports
import sys
import csv
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

# Add project root to path
project_root = Path.cwd().parent.parent.resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.anomaly_detection.models import GLCMaeV9, ReconstructionModels
from app.anomaly_detection.inference.recon_v9.local import (
    Reconf9LocalPreResult,
    Reconf9LocalThresholdCreator,
    Reconf9LocalApplier,
)

print("All imports OK")
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

## 1. Load Trained Model Checkpoint

Load the model state_dict + config saved by the training notebook (Training-GLCReconV9.ipynb).
Includes GlobalSeqNorm `running_mean`/`running_var` in state_dict.

In [ ]:
CHECKPOINT_PATH = project_root / "checkpoints" / "glc_mae_v9" / "best_glc_mae_v9.pt"

if not CHECKPOINT_PATH.exists():
    print(f"Checkpoint not found: {CHECKPOINT_PATH}")
    print("Run Training-GLCReconV9.ipynb first to train and save a model.")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

    metadata = checkpoint["metadata"]
    config_dict = checkpoint["config"]
    state_dict = checkpoint["model_state_dict"]
    robot_codebook = checkpoint.get("robot_to_idx", {})
    program_codebook = checkpoint.get("program_to_idx", {})

    model = GLCMaeV9(**config_dict)
    model.load_state_dict(state_dict)
    model.to(device).eval()

    print(f"Model restored to {device}: {model.name}")
    print(f"  Params:            {sum(p.numel() for p in model.parameters()):,}")
    print(f"  num_sensors:       {model.num_sensors}")
    print(f"  seq_len:           {model.seq_len}")
    print(f"  normalizer:        {metadata.get('normalizer', 'N/A')}")
    print(f"  GlobalSeqNorm running_mean: {model.seq_norm.bn.running_mean}")
    print(f"  GlobalSeqNorm running_var:  {model.seq_norm.bn.running_var[:3]}... (first 3)")
    print(f"  Checkpoint epoch:  {checkpoint.get('epoch', 'N/A')}")
    print(f"  Best val_loss:     {checkpoint.get('val_loss', 'N/A'):.4f}")

## 2. Load Data Files

Load threshold_calibration (for threshold selection) and test (for evaluation) data.
Both generated by `Sprint20-Generate-Data.ipynb`.

In [ ]:
DATA_DIR = project_root / "data" / "generated_sprint20"
CALIB_CSV = DATA_DIR / "threshold_calibration_welding_robot.csv"
CALIB_LABELS_CSV = DATA_DIR / "threshold_calibration_labels.csv"
TEST_CSV = DATA_DIR / "test_welding_robot.csv"
TEST_LABELS_CSV = DATA_DIR / "test_labels.csv"

SENSOR_COLS = ["wire_feed_speed_mm_min", "wire_temperature_c", "welding_current_a"]
SEQ_LEN = model.seq_len
STRIDE = 64  # no tail window

def load_csv_to_file_records(csv_path):
    """Load CSV -> group by cycle_id -> build file_records (full signal)."""
    print(f"Reading {csv_path}...")
    df = pd.read_csv(csv_path)
    print(f"  Rows: {len(df):,}, cycle_ids: {df['cycle_id'].nunique()}")
    records = []
    for cid, group in df.groupby("cycle_id"):
        group = group.sort_values("serial_number")
        file_arr = np.array([group[col].values for col in SENSOR_COLS], dtype=np.float32)
        records.append({
            "cycle_id": cid,
            "window": file_arr,
            "robot_code": str(group["robot_code"].iloc[0]),
            "program_number": int(group["program_number"].iloc[0]),
            "anomaly_type": str(group["anomaly_type"].iloc[0]),
        })
    return records

print("=" * 50)
print("LOADING DATA")
print("=" * 50)

calib_records = load_csv_to_file_records(CALIB_CSV)
test_records = load_csv_to_file_records(TEST_CSV)

# Extend codebooks for any unseen codes
for r in calib_records + test_records:
    rc, pn = str(r["robot_code"]), str(r["program_number"])
    if rc not in robot_codebook:
        robot_codebook[rc] = len(robot_codebook)
    if pn not in program_codebook:
        program_codebook[pn] = len(program_codebook)

# Load labels
calib_labels = pd.read_csv(CALIB_LABELS_CSV)
test_labels = pd.read_csv(TEST_LABELS_CSV)

calib_norm = sum(1 for r in calib_records if r["anomaly_type"] == "normal")
calib_anom = sum(1 for r in calib_records if r["anomaly_type"] != "normal")
test_norm  = sum(1 for r in test_records if r["anomaly_type"] == "normal")
test_anom  = sum(1 for r in test_records if r["anomaly_type"] != "normal")
print(f"\nThreshold Calibration: {len(calib_records)} files ({calib_norm} normal, {calib_anom} anomaly)")
print(f"Test:                  {len(test_records)} files ({test_norm} normal, {test_anom} anomaly)")

## 3. Build Macro-Window Records

V9: No tail window. Only `range(0, T - seq_len + 1, stride)`.
For T=512, seq_len=256, stride=64: exactly 5 windows.

In [ ]:
def build_pre_result_records(records_list, seq_len=None, stride=STRIDE):
    """Convert file_records -> pre_result_records for score_batch().

    V9: NO tail window. Only stride-based windows.
    """
    if seq_len is None:
        seq_len = model.seq_len
    pre_records = []
    for rec in records_list:
        file_arr = rec["window"]  # [C, T]
        C, T = file_arr.shape
        rc = rec["robot_code"]
        pn = str(rec["program_number"])

        windows = []
        serials = []
        if T >= seq_len:
            # V9: no tail window
            for start in range(0, T - seq_len + 1, stride):
                windows.append(file_arr[:, start:start + seq_len])
                serials.append(start)
        else:
            # Short file: pad to seq_len
            pad = np.zeros((C, seq_len - T), dtype=np.float32)
            windows.append(np.hstack([file_arr, pad]))
            serials.append(0)

        pre_records.append({
            "robot_code": rc,
            "program_number": pn,
            "start_time": str(rec.get("cycle_id", 0)),
            "windows": np.stack(windows, axis=0).astype(np.float32),
            "serials": np.array(serials, dtype=np.int64),
            "anomaly_type": rec["anomaly_type"],
        })
    return pre_records

calib_pre = build_pre_result_records(calib_records)
test_pre = build_pre_result_records(test_records)
print(f"Calibration records: {len(calib_pre)} files")
print(f"  Example windows shape: {calib_pre[0]['windows'].shape}")
print(f"  Example serials:       {calib_pre[0]['serials']}")
print(f"Test records: {len(test_pre)} files")

## 4. Score Files with Reconf9LocalPreResult

Uses p99 aggregation with k_skip=1.
GlobalSeqNorm stats from checkpoint are used in eval mode.

In [ ]:
engine = Reconf9LocalPreResult(
    model_version=ReconstructionModels.GLC_MAE_V9,
    metadata=metadata,
    state_dict=state_dict,
    quantile=0.99,
    k_skip=1,
    device=str(device),
    robot_codebook=robot_codebook,
    program_codebook=program_codebook,
)

print("Scoring threshold calibration set...")
calib_scores = engine.score_batch(calib_pre)
print(f"  {len(calib_scores)} files scored")

print("Scoring test set...")
test_scores = engine.score_batch(test_pre)
print(f"  {len(test_scores)} files scored")

# Show sample score keys
print(f"\nSample calibration scores[0] keys:")
for k, v in calib_scores[0].items():
    if isinstance(v, (float, int, str)):
        print(f"  {k}: {v}")
    elif isinstance(v, list) and len(v) < 10:
        print(f"  {k}: {v[:3]}... (len={len(v)})")

## 5. Threshold Calibration

Plot score distribution per sensor × per anomaly type using calibration data.
Select threshold manually (or use MAD-based automated threshold).

In [ ]:
# Separate into normal and anomaly groups
calib_norm_scores = [s for s, rec in zip(calib_scores, calib_pre) if rec["anomaly_type"] == "normal"]
all_anomaly_types = sorted(set(rec["anomaly_type"] for rec in calib_pre if rec["anomaly_type"] != "normal"))

calib_by_type = {}
for at in all_anomaly_types:
    calib_by_type[at] = [s for s, rec in zip(calib_scores, calib_pre) if rec["anomaly_type"] == at]

print("Calibration data breakdown:")
print(f"  Normal:   {len(calib_norm_scores)} files")
for at, scores in calib_by_type.items():
    print(f"  {at:15s}: {len(scores)} files")

# Plot per-type p99_score distributions
score_keys = ["p99_score", "p99_score_wire_feed_speed", "p99_score_wire_temperature", "p99_score_welding_current"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, sk in enumerate(score_keys):
    ax = axes[idx]
    norm_vals = [s[sk] for s in calib_norm_scores]
    ax.hist(norm_vals, bins=20, alpha=0.7, color="gray", label=f"normal (n={len(norm_vals)})")
    for at in all_anomaly_types:
        vals = [s[sk] for s in calib_by_type[at]]
        if vals:
            ax.hist(vals, bins=20, alpha=0.5, label=f"{at} (n={len(vals)})")
    ax.set_title(sk, fontsize=10)
    ax.set_xlabel("Score")
    ax.set_ylabel("Count")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle("Threshold Calibration: Score Distribution per Sensor × Anomaly Type", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Select Threshold & Save Config

Based on the plots above, select a threshold and save to `threshold_config.json`.
The threshold should separate normal scores (low) from anomaly scores (high).

In [ ]:
# Use MAD-based threshold from normal calibration scores
tc = Reconf9LocalThresholdCreator(quantile=0.99)
tc.fit_from_pre_results(calib_norm_scores)

print("Calibrated thresholds (MAD-based, k=3):")
for key, thresh in tc.thresholds.items():
    print(f"  {key}: {thresh:.4f}  (median={tc._medians.get(key, 0):.4f}, mad={tc._mads.get(key, 0):.4f})")

# Save threshold config
THRESHOLD_CONFIG = {
    "global_threshold": tc.thresholds.get("p99_score", 0.5),
    "score_thresholds": tc.thresholds,
    "quantile": 0.99,
    "sensors": {
        "wire_feed_speed": tc.thresholds.get("p99_score_wire_feed_speed", 0.5),
        "wire_temperature": tc.thresholds.get("p99_score_wire_temperature", 0.5),
        "welding_current": tc.thresholds.get("p99_score_welding_current", 0.5),
    },
}

import json
config_path = project_root / "config" / "threshold_config_v9.json"
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, "w") as f:
    json.dump(THRESHOLD_CONFIG, f, indent=2)
print(f"\nThreshold config saved to: {config_path}")

## 7. Evaluate on Test Set

Apply threshold to test set and compute per-type metrics.

In [ ]:
applier = Reconf9LocalApplier(THRESHOLD_CONFIG, quantile=0.99)
test_results = applier.apply(test_scores)

# Evaluate per-type
from collections import Counter
anomaly_types = [rec["anomaly_type"] for rec in test_pre]
verdicts = [r["is_file_anomaly"] for r in test_results]

print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)
print(f"{'Type':20s} {'Total':>6s} {'Flagged':>7s} {'Rate':>8s}")
print("-" * 45)
for at in sorted(set(anomaly_types)):
    indices = [i for i, t in enumerate(anomaly_types) if t == at]
    total = len(indices)
    flagged = sum(verdicts[i] for i in indices)
    rate = flagged / total * 100 if total > 0 else 0
    print(f"{at:20s} {total:6d} {flagged:7d} {rate:7.1f}%")

# Overall
total_norm = sum(1 for t in anomaly_types if t == "normal")
total_anom = len(anomaly_types) - total_norm
norm_flagged = sum(verdicts[i] for i, t in enumerate(anomaly_types) if t == "normal")
anom_flagged = sum(verdicts[i] for i, t in enumerate(anomaly_types) if t != "normal")
fpr = norm_flagged / total_norm * 100 if total_norm > 0 else 0
tpr = anom_flagged / total_anom * 100 if total_anom > 0 else 0
print("-" * 45)
print(f"{'Normal':20s} {total_norm:6d} {norm_flagged:7d} {fpr:7.1f}% (FPR)")
print(f"{'Anomaly (all)':20s} {total_anom:6d} {anom_flagged:7d} {tpr:7.1f}% (TPR)")
print("=" * 60)

## 8. Summary Report

In [ ]:
print("=" * 60)
print("GLCMaeV9 INFERENCE SUMMARY")
print("=" * 60)
print(f"Model:              GLCMaeV9 ({model.name})")
print(f"Normalizer:         log+global_seq_norm")
print(f"Aggregation:        p99 (quantile=0.99)")
print(f"Tail window:        NO (removed in V9)")
print(f"Stride:             {STRIDE}")
print(f"k_skip:             {engine.k_skip}")
print(f"Seq len:            {SEQ_LEN}")
print()
print(f"Threshold source:   threshold_calibration_welding_robot.csv "
      f"({len(calib_records)} files)")
print(f"Test data:          test_welding_robot.csv ({len(test_records)} files)")
print(f"Global threshold:   {THRESHOLD_CONFIG['global_threshold']:.4f}")
print()
print(f"Normal FPR:         {fpr:.1f}% ({norm_flagged}/{total_norm})")
print(f"Anomaly TPR:        {tpr:.1f}% ({anom_flagged}/{total_anom})")
print()
print("Per-type detection rates:")
for at in sorted(set(anomaly_types)):
    indices = [i for i, t in enumerate(anomaly_types) if t == at]
    total = len(indices)
    flagged = sum(verdicts[i] for i in indices)
    rate = flagged / total * 100 if total > 0 else 0
    print(f"  {at:20s}: {flagged:3d}/{total:<3d} ({rate:5.1f}%)")
print("=" * 60)
print("GlobalSeqNorm running stats accumulated during training,")
print("loaded from checkpoint, frozen in eval mode.")
print(f"  running_mean: {model.seq_norm.bn.running_mean}")
print(f"  running_var:  {model.seq_norm.bn.running_var[:3]}...")